# 기존 label_fix_v1 모델 평가와 실제 사진 추론
학습 코드는 없습니다. 모델 ZIP의 기존 results.csv는 과거 학습 기록이며 새 평가 결과가 아닙니다. 데이터셋이 없으면 정량 평가를 할 수 없습니다. 업로드 ZIP 안 train 사진은 학습에 사용됐으므로 독립 성능 평가에 사용하지 않습니다.

In [ ]:
from pathlib import Path
import json
import torch
import ultralytics
from ultralytics import YOLO
from PIL import Image, ImageOps
from IPython.display import display
ASSETS = Path('/workspace/recycling_assets')
WEIGHTS = ASSETS / 'label_fix_v1/weights/best.pt'
# 실제 starter 또는 준비된 데이터 폴더의 data.yaml 경로를 지정하세요.
DATA_YAML = ASSETS / 'prepared_label_fix_v1/recycling_yolo_starter/data.yaml'
OUTPUT = ASSETS / 'evaluation_outputs'
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
assert ultralytics.__version__ == '8.4.182'
assert WEIGHTS.is_file(), f'실제 모델 필요: {WEIGHTS}'
model = YOLO(str(WEIGHTS))
print('클래스:', model.names, '장치:', DEVICE)

## 1. 검증 또는 테스트 세트 평가
평가 데이터가 실제로 준비된 뒤 실행하세요. `test`는 최종 일반화 평가, `val`은 기존 검증 결과 비교용입니다. `data.yaml`의 path가 `/content`를 가리키면 원본을 수정하지 않고 여기서 로컬 평가용 사본을 만듭니다. 원본 val 기본 confidence를 유지하며 사진 표시용 conf=0.25를 평가에 강제하지 않습니다.

In [ ]:
import yaml
assert DATA_YAML.is_file(), f'평가 데이터셋/data.yaml 필요: {DATA_YAML}'
SPLIT = 'test'  # 기존 검증 로그와 비교하려면 'val'
assert SPLIT in {'val', 'test'}
cfg = yaml.safe_load(DATA_YAML.read_text(encoding='utf-8'))
DATA_ROOT = DATA_YAML.parent.resolve()  # 실제 images/labels를 포함하는 루트로 조정
cfg['path'] = str(DATA_ROOT)
assert cfg.get(SPLIT), f'{SPLIT} 경로 없음'
# 이 노트북은 원본과 같은 images/{split} 폴더 레이아웃을 사용합니다.
assert cfg[SPLIT] == f'images/{SPLIT}', '데이터 레이아웃을 확인하세요.'
folder = DATA_ROOT / cfg[SPLIT]
images = [p for p in folder.glob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
assert len(images) == 90, f'원래 평가 세트는 90장입니다: {len(images)}'
for image in images:
    assert (DATA_ROOT / 'labels' / SPLIT / (image.stem + '.txt')).is_file()
OUTPUT.mkdir(parents=True, exist_ok=True)
local_yaml = OUTPUT / 'evaluation_data.yaml'
local_yaml.write_text(yaml.safe_dump(cfg, allow_unicode=True), encoding='utf-8')
metrics = model.val(data=str(local_yaml), split=SPLIT, imgsz=640, batch=8,
                    device=DEVICE, project=str(OUTPUT), name=f'label_fix_v1_{SPLIT}',
                    exist_ok=False)
print(metrics.results_dict)
print('mAP50:', metrics.box.map50, 'mAP50-95:', metrics.box.map)
print('이번 평가 저장 위치:', metrics.save_dir)

## 2. 별도 실제 사진 추론
PHOTOS에 테스트할 사진 경로를 직접 넣으세요. 결과는 시각 확인용이며 정답 라벨 없이 정확도나 mAP를 계산할 수 없습니다. EXIF 회전을 보정합니다.

In [ ]:
PHOTOS = [
    # Path('/workspace/attachments/.../새로운_사진.jpg'),
]
if not PHOTOS:
    raise FileNotFoundError('독립적인 실제 테스트 사진을 PHOTOS에 지정하세요.')
assert all(Path(p).is_file() for p in PHOTOS)
photo_run = OUTPUT / 'real_photos'
assert not photo_run.exists(), '새 출력 폴더를 지정하세요. 이전 결과를 보존합니다.'
photo_run.mkdir(parents=True)
records = []
for index, path in enumerate(map(Path, PHOTOS)):
    with Image.open(path) as raw:
        photo = ImageOps.exif_transpose(raw).convert('RGB')
    result = model.predict(photo, conf=0.25, imgsz=640, device=DEVICE, verbose=False)[0]
    detections = [{'class_id': int(box.cls.item()),
                   'class_name': result.names[int(box.cls.item())],
                   'confidence': float(box.conf.item()),
                   'xyxy': box.xyxy[0].tolist()} for box in result.boxes]
    print(path.name, detections)
    annotated = Image.fromarray(result.plot()[:, :, ::-1])
    annotated.save(photo_run / f'{index:03d}_{path.stem}_prediction.jpg')
    preview = annotated.copy()
    preview.thumbnail((900, 900))
    display(preview)
    records.append({'source': str(path), 'detections': detections})
(photo_run / 'predictions.json').write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')